In [29]:
import pandas as pd
import matplotlib as plt

## Folder

In [ ]:
import os

OUT_DIR = "analysis"
LOG_FILES = "data/20260417/1MHz_25percent.log"
# LOG_FILES = [
#     "data/20260417/1MHz_25percent.log",
# ]

## Unit conversion

In [31]:
# convert to unit byte (kilobyte, megabyte, gigabyte)
byte_units = {
    "Bytes"   : 1, 
    "KBytes"  : 1024,
    "MBytes"  : 1024*1024,
    "GBytes"  : 1024*1024*1024
}

# convert to unit bits/sec (kilobits, megabits, gigabits)
bit_units = {
    "bits/sec"  : 1,
    "Kbits/sec" : 1e3,
    "Mbits/sec" : 1e6,
    "Gbits/sec" : 1e9
}
# for the Transfer col
def to_bytes (val, unit):
    return val * byte_units[unit]
# for the Bitrate col
def to_mbits (val, unit):
    return val * bit_units[unit] / 1e6

## Regex processors

In [32]:
import re 

# "Server listening on 5201 (test #n)"
TEST_HEADER_RE = re.compile(r"Server listening on \d+ \(test #(\d+)\)")

# [  6]   0.00-1.00   sec  4.25 MBytes  35.6 Mbits/sec    8   35.4 KBytes
# [SUM]   0.00-1.00   sec  17.0 MBytes   142 Mbits/sec   51
INTERVAL_RE = re.compile(
    r"^\[\s*(?P<id>\d+|SUM)\]\s+"
    r"(?P<start>\d+\.\d+)-(?P<end>\d+\.\d+)\s+sec\s+"
    r"(?P<xfer_val>[\d.]+)\s+(?P<xfer_unit>[KMG]?Bytes)\s+"
    r"(?P<rate_val>[\d.]+)\s+(?P<rate_unit>[KMG]?bits/sec)"
    r"(?:\s+(?P<retr>\d+))?"
    r"(?:\s+(?P<cwnd_val>[\d.]+)\s+(?P<cwnd_unit>[KMG]?Bytes))?"
)

# [  6]   0.00-30.01  sec  63.8 MBytes  17.8 Mbits/sec  243            sender
FINAL_RE = re.compile(
    r"^\[\s*(?P<id>\d+|SUM)\]\s+"
    r"(?P<start>\d+\.\d+)-(?P<end>\d+\.\d+)\s+sec\s+"
    r"(?P<xfer_val>[\d.]+)\s+(?P<xfer_unit>[KMG]?Bytes)\s+"
    r"(?P<rate_val>[\d.]+)\s+(?P<rate_unit>[KMG]?bits/sec)"
    r"(?:\s+(?P<retr>\d+))?"
    r"\s+(?P<role>sender|receiver)\s*$"
)

## Parser

In [33]:
intervals = []
finals = []
current_test = None

with open(LOG_FILES, "r") as f:
    for line in f: 
        line = line.strip()

        m = TEST_HEADER_RE.match(line)
        if m:
            current_test = int(m.group(1))
            continue
        if current_test is None:
            continue
        
        m = FINAL_RE.match(line)
        if m:
            finals.append({
                "test_id": current_test,
                "stream_id": m.group("id"),
                "start_s": float(m.group("start")),
                "end_s": float(m.group("end")),
                "transfer_bytes": to_bytes(float(m.group("xfer_val")), m.group("xfer_unit")),
                "bitrate_mbps": to_mbits(float(m.group("rate_val")), m.group("rate_unit")),
                "retr": int(m.group("retr")) if m.group("retr") else None,
                "cwnd_bytes": None,
                "kind": "final",
                "role": m.group("role"),
            })
            continue

        m = INTERVAL_RE.match(line)
        if m:
            cwnd = to_bytes(float(m.group("cwnd_val")), m.group("cwnd_unit")) \
                   if m.group("cwnd_val") else None
            intervals.append({
                "test_id": current_test,
                "stream_id": m.group("id"),
                "start_s": float(m.group("start")),
                "end_s": float(m.group("end")),
                "transfer_bytes": to_bytes(float(m.group("xfer_val")), m.group("xfer_unit")),
                "bitrate_mbps": to_mbits(float(m.group("rate_val")), m.group("rate_unit")),
                "retr": int(m.group("retr")) if m.group("retr") else None,
                "cwnd_bytes": cwnd,
                "kind": "interval",
                "role": None,
            })

df_intervals = pd.DataFrame(intervals)
df_finals = pd.DataFrame(finals)

df_intervals
# df_finals

,test_id,stream_id,start_s,end_s,transfer_bytes,bitrate_mbps,retr,cwnd_bytes,kind,role
0,1,6,0.0,1.00,4456448.00,35.6,8,36249.6,interval,None
1,1,9,0.0,1.00,2359296.00,18.9,9,25088.0,interval,None
2,1,11,0.0,1.00,3019898.88,24.1,13,16691.2,interval,None
3,1,13,0.0,1.00,3795845.12,30.4,11,16691.2,interval,None
4,1,15,0.0,1.00,4194304.00,33.5,10,23654.4,interval,None
...,...,...,...,...,...,...,...,...,...,...
727,4,9,30.0,30.01,0.00,0.0,0,11161.6,interval,None
728,4,11,30.0,30.01,262144.00,212.0,0,18124.8,interval,None
729,4,13,30.0,30.01,262144.00,212.0,0,41779.2,interval,None
730,4,15,30.0,30.01,0.00,0.0,0,40448.0,interval,None
